# 💧 AquaIntel — Interactive Exploration Notebook
**Ganges-Brahmaputra Basin Water Scarcity Analysis**

This notebook lets you:
- Explore the dataset interactively
- Visualise T-GNN graph structure
- Plot Mamba forecast outputs
- Inspect conformal prediction intervals
- Render the NOTEARS causal DAG
- Run counterfactual simulations
- Analyse MARL training curves
- Generate publication-quality figures

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette('husl')

from utils.helpers import load_config, set_seed
from data.download_datasets import load_processed, NODES, FEATURES

cfg     = load_config('../config.yaml')
set_seed(42)

dataset = load_processed()
data    = dataset['data']    # (T, N, F)
cwsi    = dataset['cwsi']    # (T, N)
dates   = pd.to_datetime(dataset['dates'])
adj     = dataset['adj']     # (N, N)
nodes   = NODES
features = FEATURES

T, N, F = data.shape
print(f'Dataset: {T} months × {N} nodes × {F} features')
print(f'Date range: {dates[0].date()} → {dates[-1].date()}')
print(f'CWSI stats: mean={cwsi.mean():.3f}, std={cwsi.std():.3f}, max={cwsi.max():.3f}')

## 1. Basin CWSI Heatmap Over Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap: time × node
node_names = [nodes[i]['name'] for i in range(N)]
im = axes[0].imshow(cwsi.T, aspect='auto', cmap='RdYlGn_r',
                    vmin=0, vmax=1, interpolation='nearest')
axes[0].set_yticks(range(N))
axes[0].set_yticklabels(node_names, fontsize=8)
# Sample date labels
tick_idx = np.linspace(0, T-1, 10, dtype=int)
axes[0].set_xticks(tick_idx)
axes[0].set_xticklabels([dates[i].strftime('%Y') for i in tick_idx], rotation=45)
axes[0].set_title('Water Stress Index (CWSI) — All Nodes × Time', fontsize=13)
plt.colorbar(im, ax=axes[0], label='CWSI')

# Mean CWSI trend
mean_cwsi = cwsi.mean(axis=1)
axes[1].fill_between(dates, mean_cwsi, alpha=0.3, color='#e63946')
axes[1].plot(dates, mean_cwsi, color='#e63946', lw=1.5, label='Basin mean CWSI')
axes[1].axhline(0.55, ls='--', color='orange', alpha=0.7, label='Moderate threshold')
axes[1].axhline(0.75, ls='--', color='red',    alpha=0.7, label='Crisis threshold')
# Shade drought years
for yr in [2009, 2014, 2023]:
    mask = dates.year == yr
    if mask.any():
        axes[1].axvspan(dates[mask][0], dates[mask][-1],
                        alpha=0.15, color='red', label=f'Drought {yr}' if yr==2009 else '')
axes[1].set_ylabel('CWSI'); axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=9); axes[1].set_title('Basin-level Water Stress Trend')
plt.tight_layout(); plt.show()

## 2. Spatial Graph Structure (T-GNN Input)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Adjacency matrix
im2 = axes[0].imshow(adj, cmap='Blues', vmin=0, vmax=1)
axes[0].set_xticks(range(N)); axes[0].set_yticks(range(N))
axes[0].set_xticklabels([nodes[i]['name'][:8] for i in range(N)],
                         rotation=90, fontsize=7)
axes[0].set_yticklabels([nodes[i]['name'][:8] for i in range(N)], fontsize=7)
axes[0].set_title('Spatial Adjacency Matrix (Graph Edges)')
plt.colorbar(im2, ax=axes[0], label='Edge weight')

# Network graph
G = nx.from_numpy_array(adj, create_using=nx.DiGraph())
node_types = [nodes[i]['type'] for i in range(N)]
type_colors = {
    'river': '#1e90ff', 'groundwater': '#8a2be2', 'urban': '#ff4500',
    'industrial': '#ff8c00', 'delta': '#00ced1', 'arid': '#daa520',
    'himalayan': '#7fff00', 'brahmaputra': '#00fa9a',
    'barrage': '#ff69b4', 'reservoir': '#00bfff', 'canal': '#ffd700'
}
node_colors = [type_colors.get(t, 'white') for t in node_types]
pos = {i: (nodes[i]['lon'] - 73, nodes[i]['lat'] - 21) for i in range(N)}

# Node sizes scaled by current CWSI
node_sizes = [300 + 800 * cwsi[-1, i] for i in range(N)]
nx.draw(G, pos=pos, ax=axes[1],
        node_color=node_colors, node_size=node_sizes,
        edge_color='white', alpha=0.8, width=0.5, arrows=True,
        arrowsize=8, labels={i: nodes[i]['name'][:6] for i in range(N)},
        font_size=6, font_color='white')
axes[1].set_title('Ganges-Brahmaputra Basin Graph\n(Node size = current CWSI)')
axes[1].set_facecolor('#0e1117')
plt.tight_layout(); plt.show()
print('Node type colour legend:', type_colors)

## 3. Feature Correlation & Seasonality

In [ ]:
# Basin-mean feature time series
data_mean = data.mean(axis=1)  # (T, F)
df = pd.DataFrame(data_mean, columns=features, index=dates)
df['cwsi'] = cwsi.mean(axis=1)

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()
colors = plt.cm.Set2(np.linspace(0, 1, F+1))

for i, feat in enumerate(features):
    axes[i].plot(dates, df[feat], color=colors[i], lw=1)
    axes[i].fill_between(dates, df[feat], alpha=0.2, color=colors[i])
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_ylabel('')
    axes[i].tick_params(labelsize=7)

plt.suptitle('Basin-Mean Feature Trends (2002–2024)', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

# Correlation with CWSI
corrs = df.corr()['cwsi'].drop('cwsi').sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
colors_corr = ['#e63946' if v > 0 else '#2dc653' for v in corrs]
ax.barh(corrs.index, corrs.values, color=colors_corr)
ax.axvline(0, color='white', lw=0.8)
ax.set_title('Pearson Correlation with CWSI'); ax.set_xlabel('Correlation')
plt.tight_layout(); plt.show()

## 4. Run & Inspect T-GNN (Module 1)

In [ ]:
# Quick training run (reduce epochs for notebook)
cfg_nb = load_config('../config.yaml')
cfg_nb['tgnn']['epochs'] = 10   # increase to 100 for full training

from modules.module1_tgnn import train_tgnn
tgnn_model, embeddings, metrics = train_tgnn(dataset, cfg_nb)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(metrics['train_losses'], label='Train', color='#00b4d8')
ax1.plot(metrics['val_losses'],   label='Val',   color='#f4a261')
ax1.set_title('T-GNN Training Curve'); ax1.legend()
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Huber Loss')

# Embedding PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
# Last timestep embeddings for all nodes
emb_2d = pca.fit_transform(embeddings[-1])
scatter = ax2.scatter(emb_2d[:, 0], emb_2d[:, 1],
                      c=cwsi[-1], cmap='RdYlGn_r', s=80, vmin=0, vmax=1)
for i, name in enumerate([nodes[j]['name'] for j in range(N)]):
    ax2.annotate(name[:7], emb_2d[i], fontsize=7, ha='center')
plt.colorbar(scatter, ax=ax2, label='CWSI')
ax2.set_title(f'T-GNN Embedding PCA (expl. var: {pca.explained_variance_ratio_.sum():.1%})')
plt.tight_layout(); plt.show()

## 5. Mamba Forecast Visualization (Module 2)

In [ ]:
cfg_nb['mamba']['epochs'] = 10  # increase for full training

from modules.module2_mamba_forecast import train_mamba
mamba_model, mamba_metrics = train_mamba(dataset, cfg_nb, embeddings)

# Plot training curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mamba_metrics['train_losses'], label='Train', color='#00b4d8')
ax.plot(mamba_metrics['val_losses'],   label='Val',   color='#f4a261')
ax.set_title('Mamba SSM Training — Quantile Loss'); ax.legend()
ax.set_xlabel('Epoch'); ax.set_ylabel('Pinball Loss')
plt.tight_layout(); plt.show()

print('Mamba training complete!')
print(f'Best val loss: {min(mamba_metrics["val_losses"]):.4f}')

## 6. NOTEARS Causal DAG (Module 4)

In [ ]:
from modules.module4_causal import run_causal_module
causal = run_causal_module(dataset, cfg)

W_est    = causal['W_est']
var_names = causal['variable_names']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Weight matrix
im = axes[0].imshow(W_est, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xticks(range(len(var_names)))
axes[0].set_yticks(range(len(var_names)))
axes[0].set_xticklabels([v[:10] for v in var_names], rotation=90, fontsize=8)
axes[0].set_yticklabels([v[:10] for v in var_names], fontsize=8)
axes[0].set_title('Causal Weight Matrix W_est\n(red=positive, blue=negative causal effect)')
plt.colorbar(im, ax=axes[0])

# Causal DAG as network
G_causal = nx.from_numpy_array(W_est, create_using=nx.DiGraph())
mapping   = {i: var_names[i][:12] for i in range(len(var_names))}
G_causal  = nx.relabel_nodes(G_causal, mapping)

# Colour the target node
n_colors = ['#e63946' if 'water_stress' in v else '#1e90ff' for v in G_causal.nodes]
pos_c    = nx.spring_layout(G_causal, seed=42, k=2)
edge_w   = [abs(G_causal[u][v]['weight']) * 3 for u, v in G_causal.edges()]
edge_c   = ['#e63946' if G_causal[u][v]['weight'] > 0 else '#2dc653'
             for u, v in G_causal.edges()]
nx.draw(G_causal, pos=pos_c, ax=axes[1],
        node_color=n_colors, node_size=800,
        edge_color=edge_c, width=edge_w, arrows=True,
        arrowsize=15, with_labels=True, font_size=7, font_color='white')
axes[1].set_title('Learned Causal DAG (NOTEARS)\nRed=positive, Green=negative causal edge')
axes[1].set_facecolor('#0e1117')
plt.tight_layout(); plt.show()

print('\nCausal ranking (top drivers of water stress):')
print(causal['causal_ranking'].to_string(index=False))

## 7. Conformal Prediction Intervals (Module 3)

In [ ]:
from modules.module3_conformal import run_conformal_module
conf_results = run_conformal_module(dataset, cfg, mamba_model, embeddings)

y_true  = conf_results['test_true']
y_lo    = conf_results['conformal_lower']
y_hi    = conf_results['conformal_upper']
n_plot  = min(200, len(y_true))

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(n_plot)
ax.fill_between(x, y_lo[:n_plot], y_hi[:n_plot],
                alpha=0.3, color='#00b4d8', label='90% Conformal Interval')
ax.plot(x, y_true[:n_plot], 'w-', lw=0.8, label='True CWSI', alpha=0.8)
ax.axhline(0.75, ls='--', color='red', lw=0.8, label='Crisis threshold')

# Mark out-of-interval points
miss = (y_true[:n_plot] < y_lo[:n_plot]) | (y_true[:n_plot] > y_hi[:n_plot])
ax.scatter(x[miss], y_true[:n_plot][miss], color='red', s=15,
           zorder=5, label=f'Outside interval ({miss.sum()} points)')
ax.set_ylim(0, 1); ax.legend(fontsize=9)
ax.set_title('Conformal Prediction Intervals (Test Set)')
ax.set_xlabel('Test sample index'); ax.set_ylabel('CWSI')
plt.tight_layout(); plt.show()

metrics = conf_results['interval_metrics']
print(f"\nConformal Coverage: {metrics['Coverage']:.3f}  (target ≥ 0.90)")
print(f"Mean Interval Width: {metrics['Width']:.4f}")
print(f"Coverage-Width Criterion (lower=better): {metrics['CWC']:.4f}")

## 8. MARL Training & Evaluation (Module 5)

In [ ]:
cfg_nb['marl']['train_steps'] = 10000  # increase for full training

from modules.module5_marl import run_marl_module
marl = run_marl_module(dataset, cfg_nb)

rewards = marl['train_results']['episode_rewards']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Smoothed reward
smooth = pd.Series(rewards).rolling(20, min_periods=1).mean()
axes[0].plot(rewards, alpha=0.2, color='#00b4d8')
axes[0].plot(smooth,  color='#00b4d8', lw=2)
axes[0].set_title('QMIX Episode Rewards During Training')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Cumulative reward')

# Allocation metrics
ev = marl['eval_results']
metrics_names = ['Efficiency', 'Equity', 'Sustainability']
metric_vals   = [
    ev['mean_efficiency'],
    1 - ev['mean_equity_pen'],
    1 - ev['mean_sus_pen'],
]
colors_m = ['#2196F3', '#4CAF50', '#FF9800']
bars = axes[1].bar(metrics_names, metric_vals, color=colors_m, width=0.5)
axes[1].set_ylim(0, 1); axes[1].axhline(0.7, ls='--', color='white', alpha=0.4)
for bar, val in zip(bars, metric_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f'{val:.2%}', ha='center', fontsize=10)
axes[1].set_title('QMIX Policy — Evaluation Metrics')
plt.tight_layout(); plt.show()

## 9. Publication-Ready Summary Figure

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs  = fig.add_gridspec(2, 4, hspace=0.4, wspace=0.35)

# Panel A: CWSI heatmap
ax_a = fig.add_subplot(gs[0, :2])
ax_a.imshow(cwsi.T, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=1)
ax_a.set_title('(A) Water Stress Index — All Nodes × Time', fontsize=11)
ax_a.set_xlabel('Timestep'); ax_a.set_ylabel('Node')

# Panel B: Causal ranking
ax_b = fig.add_subplot(gs[0, 2:])
cr = causal['causal_ranking']
ax_b.barh(cr['Variable'][:6], cr['Causal_Strength'][:6],
          color=cm.RdYlGn_r(cr['Causal_Strength'][:6].values))
ax_b.set_title('(B) Causal Drivers of Water Stress (NOTEARS)', fontsize=11)
ax_b.set_xlabel('Causal Strength')

# Panel C: Conformal intervals (sample)
ax_c = fig.add_subplot(gs[1, :2])
n_p = min(100, len(y_true))
ax_c.fill_between(range(n_p), y_lo[:n_p], y_hi[:n_p],
                  alpha=0.3, color='#00b4d8')
ax_c.plot(range(n_p), y_true[:n_p], 'w-', lw=1)
ax_c.set_title('(C) Conformal Prediction Intervals (90% coverage)', fontsize=11)
ax_c.set_xlabel('Sample'); ax_c.set_ylabel('CWSI')
ax_c.set_ylim(0, 1)

# Panel D: MARL allocation
ax_d = fig.add_subplot(gs[1, 2:])
agents_plot = ['Agriculture', 'Industry', 'Municipal']
alloc_plot  = [ev['mean_efficiency'] * 0.5,
               ev['mean_efficiency'] * 0.2,
               ev['mean_efficiency'] * 0.3]
ax_d.bar(agents_plot, alloc_plot, color=['#2196F3','#FF9800','#4CAF50'])
ax_d.set_title('(D) QMIX Optimal Allocation (Normalised)', fontsize=11)
ax_d.set_ylabel('Allocation score')

plt.suptitle('AquaIntel: Water Scarcity ML System — Ganges-Brahmaputra Basin',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('aquaintel_summary.pdf', bbox_inches='tight', dpi=300)
plt.show()
print('Saved: aquaintel_summary.pdf')